# Skin Lesion Classification — Training Pipeline

**CNN ensemble for automated dermoscopy image classification**  
ResNet-50 · DenseNet-121 · EfficientNet-B3 · TTA · Clinical threshold calibration

---

## Before running

1. **Install dependencies** (first time only, then restart runtime)
2. **Configure paths** in the *Configuration* cell below
3. **Choose execution mode** — see the three options below
4. **Run all cells** with `Runtime → Run all`

The pipeline auto-detects Google Colab and Kaggle environments. No manual path changes are needed for standard setups.

In [ ]:
# Run once, then Runtime → Restart runtime
# If you see 'torch._utils not found': Runtime → Factory reset runtime, then run this again
!pip install "albumentations>=1.3.0,<2.0.0" -q

## Execution modes

The pipeline supports three modes. **Only one should be active at a time.**
Set the corresponding variable to `True` or a path, and leave the others as `False`.

---

### Mode 1 — Full training from scratch
```python
LOAD_TTA_FROM_DIR       = False
RESUME_FROM_CHECKPOINTS = False
```
Trains all three models from ImageNet weights. Takes ~4 hours on Colab T4 or ~2.5 hours on Kaggle P100. Generates a complete PDF report with training curves, confusion matrices, ROC curves, Grad-CAM and clinical metrics.

---

### Mode 2 — Resume from checkpoints
```python
LOAD_TTA_FROM_DIR       = False
RESUME_FROM_CHECKPOINTS = True
RESUME_DIR              = "/path/to/previous/output/folder"
FORCE_RETRAIN           = []   # or e.g. ['efficientnet_b3'] to force one model
```
Loads existing `.pth` checkpoints from a previous run. Models with a valid checkpoint are skipped — only missing ones are trained. Creates a new output folder with suffix `_r`.

**When to use:** session disconnected mid-training, or one model converged poorly and you want to retrain only that one (`FORCE_RETRAIN`).

---

### Mode 3 — Threshold recalibration only
```python
LOAD_TTA_FROM_DIR       = "/path/to/previous/output/folder"
RESUME_FROM_CHECKPOINTS = False
```
Loads the saved TTA probability arrays (`tta_sum_probs.npy`, `tta_val_sum.npy`) from a previous run. Skips all training and TTA — only recalibrates the clinical thresholds and regenerates the PDF. Completes in under 5 minutes.

**When to use:** exploring different threshold strategies without retraining.

## Configuration

Set your paths and execution mode here. This is the only cell you need to edit.

In [ ]:
# ── Dataset location ─────────────────────────────────────────────────────
# Colab:  folder name inside Google Drive MyDrive/
# Kaggle: dataset slug on Kaggle (username/dataset-name)
COLAB_DRIVE_FOLDER  = 'ISIC2018_Task3'
KAGGLE_DATASET_SLUG = 'danielortizrequena/isic2018-task3'

# ── Reproducibility ───────────────────────────────────────────────────────
# Change to 7 or 123 for robustness experiments (keep 42 for the reference run)
RNG_SEED = 42

# ── Execution mode — set only ONE at a time ───────────────────────────────

# Mode 3: threshold recalibration only (fastest — no training)
# Set to the full path of a previous output folder, or False to disable
LOAD_TTA_FROM_DIR = False
# Example: LOAD_TTA_FROM_DIR = '/content/drive/MyDrive/ISIC2018_Task3/OUTPUTS/2026-05-20_13-15-53'

# Mode 2: resume from existing checkpoints
RESUME_FROM_CHECKPOINTS = False
RESUME_DIR = ''
# Example: RESUME_DIR = '/content/drive/MyDrive/ISIC2018_Task3/OUTPUTS/2026-05-20_13-15-53'

# Force-retrain specific models even if checkpoints exist (used with Mode 2)
FORCE_RETRAIN = []   # e.g. ['efficientnet_b3']

# Mode 1: full training from scratch — active when both above are False

## Run the pipeline

The cell below loads the full pipeline script from the repository and executes it.
All configuration set above is picked up automatically.

**Outputs** are saved to `OUTPUTS/<timestamp>/` inside your Drive folder:
- `resultados_tfg.pdf` — complete results report
- `results.json` — machine-readable metrics summary
- `*_best.pth` — model checkpoints
- `tta_sum_probs.npy` / `tta_val_sum.npy` — TTA probability arrays (for Mode 3)
- `<timestamp>.zip` — full output archive
- `ejecucion_log.txt` — complete console log

In [ ]:
import os, sys

# Clone the repository to get the latest pipeline code
if not os.path.exists('/content/skin-lesion-classifier-CNN'):
    os.system('git clone https://github.com/daorre1202/skin-lesion-classifier-CNN.git '
              '/content/skin-lesion-classifier-CNN')
else:
    os.system('cd /content/skin-lesion-classifier-CNN && git pull')

# Copy pipeline script to working directory
os.system('cp /content/skin-lesion-classifier-CNN/CodigoTFG_DanielOrtiz.py '
          '/content/CodigoTFG_DanielOrtiz.py')

# Pass configuration variables to the script
import builtins
builtins._NB_COLAB_DRIVE_FOLDER   = COLAB_DRIVE_FOLDER
builtins._NB_KAGGLE_DATASET_SLUG  = KAGGLE_DATASET_SLUG
builtins._NB_RNG_SEED             = RNG_SEED
builtins._NB_LOAD_TTA_FROM_DIR    = LOAD_TTA_FROM_DIR
builtins._NB_RESUME               = RESUME_FROM_CHECKPOINTS
builtins._NB_RESUME_DIR           = RESUME_DIR
builtins._NB_FORCE_RETRAIN        = FORCE_RETRAIN

exec(open('/content/CodigoTFG_DanielOrtiz.py').read())
main()

## Results

When the pipeline finishes, find your outputs in:
```
Google Drive → MyDrive/ISIC2018_Task3/OUTPUTS/<timestamp>/
```

Key files:
- **`resultados_tfg.pdf`** — open this for a full visual summary of the run
- **`results.json`** — structured metrics (BACC, per-class sensitivity/specificity, thresholds)
- **`<timestamp>.zip`** — download this to keep a complete local copy

To compare results across seeds, see `results/robustness_summary.json` in this repository.